In [4]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.ndimage import label
from skimage import morphology
from skimage import measure
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries
from scipy.stats import gaussian_kde
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import curve_fit
from scipy.ndimage import map_coordinates
import seaborn as sns
from scipy.ndimage import maximum_filter, label
from scipy.stats import skewnorm
from scipy.stats import lognorm
import fifpy
from scipy.ndimage import gaussian_filter
from scipy.ndimage import label, gaussian_filter
from scipy.stats import lognorm, gaussian_kde
import csv
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from time import perf_counter
from matplotlib.patches import Circle, FancyArrowPatch
from skimage import morphology, measure
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.detection import detect_current_sheets
from shared.fields import partial_x_fft, partial_y_fft
from shared.filtering import iterative_filtering1D, iterative_filtering3D
from shared.thickness import _m3_fwhm, _walk_m2, gaussian
from shared.paths import DATA_ROOT, NAS_ROOT, results_dir, cache_dir as shared_cache_dir
RESULTS = results_dir("cs_identification")
CACHE = shared_cache_dir("cs_identification")

In [5]:
#Defining all the required functions

#Fourier Transform Functions


#Defining constants
dx = 0.0625
dy = 0.0625





In [4]:
# =====================================================================
#  TWO animations of the current-sheet THICKNESS distribution over time
#    (1) step HISTOGRAMS of M1/M2/M3
#    (2) LOGNORMAL FITS of M1/M2/M3
#  ------------------------------------------------------------
#  One beta. Each frame = one snapshot in TIMES. Per-frame thickness extraction
#  runs in parallel over TIMES (cached filtering reused) and is done ONCE;
#  both movies are rendered from the same frame data with FIXED axes.
#
#  M1/M2/M3 algorithms and filters are unchanged; all use ONE 8-connected label.
#  Must exist in your session:
#    iterative_filtering3D, partial_x_fft, partial_y_fft, detect_current_sheets
# =====================================================================

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import h5py
from skimage import morphology, measure
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter
from scipy.optimize import curve_fit
from scipy.stats import lognorm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

# -----------------------------
# USER SETTINGS
# -----------------------------
beta      = 1                              # one run
TIMES     = list(range(5, 300, 1))         # frames (one per snapshot)
sim_tag   = "ApJ"
base_root = str(DATA_ROOT)

cache_dir = str(CACHE / "filtered_cache_Jz")
out_hist  = str(RESULTS / f"thickness_pdf_hist_beta{beta}.mp4")       # video 1
out_fit   = str(RESULTS / f"thickness_pdf_lognorm_beta{beta}.mp4")    # video 2

PX_PER_DI    = 16.0
CONNECTIVITY = 2
N_WORKERS    = min(len(TIMES), os.cpu_count() or 1)
THREADS_PER_WORKER = 1
FPS          = 2
NBINS        = 30
YMAX_PCTILE  = 97          # y-limit = this percentile of per-frame peak heights, not the
                           # global max (a few sparse/spiky frames won't blow up the axis).
                           # Lower it (e.g. 90) if still too tall; rare spikes clip at top.
MIN_N_SCALE  = 5           # frames with fewer sheets than this don't set the y-scale
dx = dy      = 0.0625

FILTER_PARAMS_BETA = {0.0625: (3, 2.0), 0.25: (3, 2.3), 1: (3, 2.5), 4: (3, 2.7), 16: (3, 3.0)}
max_len, subpix_step, margin_frac, min_half, smooth_sigma = 60, 0.5, 0.5, 6.0, 3

os.makedirs(cache_dir, exist_ok=True)


def _init_worker(n_threads):
    if threadpool_limits is not None:
        threadpool_limits(limits=n_threads)




def filter_params(b):
    return FILTER_PARAMS_BETA[b]


# -----------------------------
# One snapshot -> (time, m1_di, m2_di, m3_di)   (no plotting; cache reused)
# -----------------------------
def extract(time):
    n_iter, Xi = filter_params(beta)
    base_path = f"{base_root}/beta_p_{beta}/beta_e_1/"
    with h5py.File(base_path + f"Bx_{sim_tag}_t{time}.h5", "r") as fBx, \
         h5py.File(base_path + f"By_{sim_tag}_t{time}.h5", "r") as fBy, \
         h5py.File(base_path + f"Bz_{sim_tag}_t{time}.h5", "r") as fBz:
        data_Bx, data_By, data_Bz = fBx["DS1"][:].T, fBy["DS1"][:].T, fBz["DS1"][:].T

    cache_file = f"{cache_dir}/filtered_beta_{beta}_t{time}_iter{n_iter}_Xi{Xi}.npz"
    if os.path.exists(cache_file):
        d = np.load(cache_file)
        data_Bx_f, data_By_f, data_Bz_f = d["Bx"], d["By"], d["Bz"]
    else:
        data_Bx_f, data_By_f, data_Bz_f = iterative_filtering3D(
            data_Bx, data_By, data_Bz, n_iter, Xi=Xi)
        np.savez_compressed(cache_file, Bx=data_Bx_f, By=data_By_f, Bz=data_Bz_f)

    Jz = partial_x_fft(data_By_f, dy) - partial_y_fft(data_Bx_f, dx)
    J2 = np.abs(Jz)

    sheets = detect_current_sheets(J2)
    mask = np.zeros_like(J2)
    for sheet in sheets:
        for (i, j) in sheet:
            mask[i, j] = 1
    struct = morphology.disk(1)
    mask = morphology.binary_opening(mask, struct)
    mask = morphology.binary_closing(mask, struct)
    cleaned_mask = morphology.remove_small_objects(mask.astype(bool), min_size=500)

    labeled = measure.label(cleaned_mask, connectivity=CONNECTIVITY)
    nreg = int(labeled.max())

    J2_smooth = gaussian_filter(J2, sigma=smooth_sigma)
    gx, gy = np.gradient(J2_smooth)

    m1_list, m2_list, m3_list = [], [], []
    for k in range(1, nreg + 1):
        region = (labeled == k)
        xs_r, ys_r = np.where(region)
        p = int(np.argmax(J2[region]))
        xm, ym = int(xs_r[p]), int(ys_r[p])

        edt = ndi.distance_transform_edt(region)
        m1_list.append(2 * float(edt.max()))

        gvec = np.array([gx[xm, ym], gy[xm, ym]])
        if np.allclose(gvec, 0):
            dx_, dy_ = 0.0, 0.0
        else:
            v = -gvec / np.linalg.norm(gvec); dx_, dy_ = float(v[0]), float(v[1])

        m2_list.append(_walk_m2(cleaned_mask, xm, ym, dx_, dy_)[0]
                       + _walk_m2(cleaned_mask, xm, ym, -dx_, -dy_)[0])
        m3_list.append(_m3_fwhm(J2, cleaned_mask, xm, ym, dx_, dy_))

    m1 = np.array(m1_list, float); m1 = m1[m1 > 0]
    m2 = np.array(m2_list, float); m2 = m2[m2 > 0]
    m3 = np.array(m3_list, float); m3 = m3[np.isfinite(m3)]; m3 = m3[(m3 > 0) & (m3 <= 100)]
    return time, m1 / PX_PER_DI, m2 / PX_PER_DI, m3 / PX_PER_DI


# -----------------------------
# Helpers for the two movies
# -----------------------------
def fit_lognorm(x):
    x = x[np.isfinite(x) & (x > 0)]
    if x.size < 5:
        return None
    shape, loc, scale = lognorm.fit(x, floc=0)     # loc fixed at 0
    return shape, scale                            # median = scale, sigma = shape


def save_anim(anim, path):
    try:
        anim.save(path, writer="ffmpeg", fps=FPS, dpi=150); print(f"Saved {path}", flush=True)
    except Exception as e:
        p = path.replace(".mp4", ".gif")
        print(f"ffmpeg failed ({e}); writing {p} ...", flush=True)
        anim.save(p, writer="pillow", fps=FPS); print(f"Saved {p}", flush=True)


# -----------------------------
if __name__ == "__main__":
    # ---- extract all frames in parallel (ONCE) ----
    results = []
    if N_WORKERS == 1:
        for t in TIMES:
            try:
                results.append(extract(t)); print(f"t={t}: done", flush=True)
            except FileNotFoundError:
                print(f"t={t}: missing file, skipped", flush=True)
    else:
        ctx = mp.get_context("fork")
        if threadpool_limits is None:
            print("note: threadpoolctl not found -> workers may oversubscribe threads.", flush=True)
        print(f"Extracting {len(TIMES)} frames across {N_WORKERS} workers ...", flush=True)
        with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx,
                                 initializer=_init_worker, initargs=(THREADS_PER_WORKER,)) as ex:
            futs = {ex.submit(extract, t): t for t in TIMES}
            for fut in as_completed(futs):
                t = futs[fut]
                try:
                    results.append(fut.result()); print(f"t={t}: done", flush=True)
                except FileNotFoundError:
                    print(f"t={t}: missing file, skipped", flush=True)

    if not results:
        raise SystemExit("No frames extracted (check paths / TIMES).")
    frames = sorted(results, key=lambda r: r[0])

    # ---- fixed axes: global log-spaced bins, fit grid, and both y-maxes ----
    allv = np.concatenate([np.concatenate([f[1], f[2], f[3]]) for f in frames])
    allv = allv[allv > 0]
    bins  = np.logspace(np.log10(allv.min()), np.log10(allv.max()), NBINS)
    xgrid = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), 400)
    series = [("M1", "C0"), ("M2", "C1"), ("M3", "C2")]

    ymax_hist = 0.0
    hist_peaks, fit_peaks = [], []
    frame_fits = []
    for _, m1, m2, m3 in frames:
        for arr in (m1, m2, m3):
            if len(arr) >= MIN_N_SCALE:
                hist_peaks.append(np.histogram(arr, bins=bins, density=True)[0].max())
        d = {}
        for arr, (name, _) in zip((m1, m2, m3), series):
            f = fit_lognorm(arr); d[name] = f
            if f is not None and len(arr[np.isfinite(arr) & (arr > 0)]) >= MIN_N_SCALE:
                fit_peaks.append(np.nanmax(lognorm.pdf(xgrid, f[0], 0, f[1])))
        frame_fits.append(d)

    # robust y-limits: a high percentile of the per-frame peaks, not the global max
    ymax_hist = (np.percentile(hist_peaks, YMAX_PCTILE) if hist_peaks else 1.0) * 1.1
    ymax_fit  = (np.percentile(fit_peaks,  YMAX_PCTILE) if fit_peaks  else 1.0) * 1.1

    # ---- VIDEO 1: histograms ----
    fig1, ax1 = plt.subplots(figsize=(8, 5))

    def draw_hist(i):
        t, m1, m2, m3 = frames[i]
        ax1.clear()
        for arr, (name, col) in zip((m1, m2, m3), series):
            if len(arr):
                ax1.hist(arr, bins=bins, density=True, histtype="step", lw=2, color=col,
                         label=f"{name}  (median {np.median(arr):.2f} $d_i$, N={len(arr)})")
                ax1.axvline(np.median(arr), color=col, ls=":", lw=1)
        ax1.set_xscale("log"); ax1.set_xlim(bins[0], bins[-1]); ax1.set_ylim(0, ymax_hist)
        ax1.set_xlabel(r"thickness $(d_i)$"); ax1.set_ylabel("PDF")
        ax1.set_title(rf"CS thickness PDF (histogram)  ($\beta_p={beta}$, $t={t}$)")
        ax1.legend(loc="upper right", fontsize=9)

    save_anim(FuncAnimation(fig1, draw_hist, frames=len(frames), interval=1000 / FPS, blit=False), out_hist)
    plt.close(fig1)

    # ---- VIDEO 2: lognormal fits ----
    fig2, ax2 = plt.subplots(figsize=(8, 5))

    def draw_fit(i):
        t = frames[i][0]
        ax2.clear()
        d = frame_fits[i]
        for name, col in series:
            f = d[name]
            if f is not None:
                shape, scale = f
                ax2.plot(xgrid, lognorm.pdf(xgrid, shape, 0, scale), "-", lw=2, color=col,
                         label=fr"{name}  (median {scale:.2f} $d_i$, $\sigma$={shape:.2f})")
                ax2.axvline(scale, color=col, ls=":", lw=1)
        ax2.set_xscale("log"); ax2.set_xlim(bins[0], bins[-1]); ax2.set_ylim(0, ymax_fit)
        ax2.set_xlabel(r"thickness $(d_i)$"); ax2.set_ylabel("PDF (lognormal fit)")
        ax2.set_title(rf"CS thickness lognormal fit  ($\beta_p={beta}$, $t={t}$)")
        ax2.legend(loc="upper right", fontsize=9)

    save_anim(FuncAnimation(fig2, draw_fit, frames=len(frames), interval=1000 / FPS, blit=False), out_fit)
    plt.close(fig2)

Extracting 295 frames across 128 workers ...
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMFIMF  11

IMF 1
IMF 1
IMF 1
IMFIMF  11

IMFIMF 1 
1
IMF 1
IMF 1
IMF 1
IMF 1
IMFIMF  1
1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMFIMF  1
1
IMF 1
IMF 1IMF
 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMFIMF 1 
1
IMF 1
IMF 1
IMFIMF 1
 1IMFIMF
  IMF11
 
1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF IMF1 
1
IMF 1
IMFIMF  11

IMF IMF1 
1
IMF 1
IMF IMF1IMFIMF IMFIMFIMF    1IMF1
11
IMF
 
1

1
IMFIMF  1
 1IMFIMF 
IMF 1 1 

1
1
1IMFIMFIMFIMF  1 11


 
IMFIMF1 1 1
IMF 1IMF
 1


IMFIMFIMFIMF  11IMF 1 

 1

IMFIMF  11
1

IMFIMF 1 1
IMF
 1
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
t=5: done
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMFIMF  2
2
IMF IMF 1
2
IMF 2
IMF IMFIMF 2
 22

IMF 2
IMFIMF  22

IMF 2
IMF 2
IMF 2IMF
 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF IMF2 
2
IMF IMF 2
2
IMF 2
IMF 2
IMF 2IMF
 2
IMF 2IMF 
2
IMFIMF  2
2
IMF 2
IMF 2
IMFIMF 2
IMF  IMF22

 2
IMF

/tmp/ipykernel_140620/2319685275.py:304: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax2.legend(loc="upper right", fontsize=9)


Saved thickness_pdf_lognorm_beta1.mp4


In [ ]:
# =====================================================================
#  ONE two-panel movie over time (starts at t = 5):
#    LEFT  : current-sheet detection  (|J_parallel| map + cyan sheet outline)
#    RIGHT : thickness distribution   (M1/M2/M3 step histograms)
#  Both panels come from a single extraction per frame (same pipeline).
#  beta_p = 0.0625, beta_e = 1.
#
#  Must exist in your session:
#    iterative_filtering3D, partial_x_fft, partial_y_fft, detect_current_sheets
# =====================================================================

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import h5py
from skimage import morphology, measure
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter
from scipy.optimize import curve_fit
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
    'figure.titlesize': 20
})

# -----------------------------
# SETTINGS
# -----------------------------
beta        = 0.0625
frame_step  = 1
TIMES       = list(range(0, 301, frame_step))   # START at t=0 (empty early frames handled below)
sim_tag     = "ApJ"
base_root   = str(DATA_ROOT)
cache_dir   = str(CACHE / "filtered_cache_Jz")
out_movie   = str(RESULTS / f"cs_detection_histogram_beta{beta}.mp4")
fps         = 6                                 # 2x slower than before (was 12)

PX_PER_DI    = 16.0
CONNECTIVITY = 2
N_WORKERS    = min(len(TIMES), os.cpu_count() or 1)
THREADS_PER_WORKER = 1
NBINS        = 30
YMAX_PCTILE  = 97          # robust y-limit for the histogram (ignore sparse/spiky frames)
MIN_N_SCALE  = 5           # frames with fewer sheets than this don't set the y-scale
dx = dy      = 0.0625

FILTER_PARAMS_BETA = {0.0625: (3, 2.0), 0.25: (3, 2.3), 1: (3, 2.5), 4: (3, 2.7), 16: (3, 3.0)}
max_len, subpix_step, margin_frac, min_half, smooth_sigma = 60, 0.5, 0.5, 6.0, 3

os.makedirs(cache_dir, exist_ok=True)


# -----------------------------
# One snapshot -> (time, |J_par| field, mask, m1_di, m2_di, m3_di)
#   detection (field + mask) feeds the LEFT panel; thicknesses feed the RIGHT.
# -----------------------------
def extract(time):
    n_iter, Xi = filter_params(beta)
    base_path = f"{base_root}/beta_p_{beta}/beta_e_1/"
    with h5py.File(base_path + f"Bx_{sim_tag}_t{time}.h5", "r") as fBx, \
         h5py.File(base_path + f"By_{sim_tag}_t{time}.h5", "r") as fBy, \
         h5py.File(base_path + f"Bz_{sim_tag}_t{time}.h5", "r") as fBz:
        data_Bx, data_By, data_Bz = fBx["DS1"][:].T, fBy["DS1"][:].T, fBz["DS1"][:].T

    cache_file = f"{cache_dir}/filtered_beta_{beta}_t{time}_iter{n_iter}_Xi{Xi}.npz"
    if os.path.exists(cache_file):
        d = np.load(cache_file)
        data_Bx_f, data_By_f, data_Bz_f = d["Bx"], d["By"], d["Bz"]
    else:
        data_Bx_f, data_By_f, data_Bz_f = iterative_filtering3D(
            data_Bx, data_By, data_Bz, n_iter, Xi=Xi)
        np.savez_compressed(cache_file, Bx=data_Bx_f, By=data_By_f, Bz=data_Bz_f)

    Jz = partial_x_fft(data_By_f, dy) - partial_y_fft(data_Bx_f, dx)
    J2 = np.abs(Jz)                                  # |J_parallel|

    sheets = detect_current_sheets(J2)
    mask = np.zeros_like(J2)
    for sheet in sheets:
        for (i, j) in sheet:
            mask[i, j] = 1
    struct = morphology.disk(1)
    mask = morphology.binary_opening(mask, struct)
    mask = morphology.binary_closing(mask, struct)
    cleaned_mask = morphology.remove_small_objects(mask.astype(bool), min_size=500)

    labeled = measure.label(cleaned_mask, connectivity=CONNECTIVITY)
    nreg = int(labeled.max())

    J2_smooth = gaussian_filter(J2, sigma=smooth_sigma)
    gx, gy = np.gradient(J2_smooth)

    m1_list, m2_list, m3_list = [], [], []
    for k in range(1, nreg + 1):
        region = (labeled == k)
        xs_r, ys_r = np.where(region)
        p = int(np.argmax(J2[region]))
        xm, ym = int(xs_r[p]), int(ys_r[p])

        edt = ndi.distance_transform_edt(region)
        m1_list.append(2 * float(edt.max()))

        gvec = np.array([gx[xm, ym], gy[xm, ym]])
        if np.allclose(gvec, 0):
            dx_, dy_ = 0.0, 0.0
        else:
            v = -gvec / np.linalg.norm(gvec); dx_, dy_ = float(v[0]), float(v[1])

        m2_list.append(_walk_m2(cleaned_mask, xm, ym, dx_, dy_)[0]
                       + _walk_m2(cleaned_mask, xm, ym, -dx_, -dy_)[0])
        m3_list.append(_m3_fwhm(J2, cleaned_mask, xm, ym, dx_, dy_))

    m1 = np.array(m1_list, float); m1 = m1[m1 > 0]
    m2 = np.array(m2_list, float); m2 = m2[m2 > 0]
    m3 = np.array(m3_list, float); m3 = m3[np.isfinite(m3)]; m3 = m3[(m3 > 0) & (m3 <= 100)]
    return (time, J2.astype(np.float32), cleaned_mask.astype(bool),
            m1 / PX_PER_DI, m2 / PX_PER_DI, m3 / PX_PER_DI)


# -----------------------------
if __name__ == "__main__":
    # ---- extract all frames in parallel (ONCE) ----
    results = []
    if N_WORKERS == 1:
        for t in TIMES:
            try:
                results.append(extract(t)); print(f"t={t}: done", flush=True)
            except FileNotFoundError:
                print(f"t={t}: missing file, skipped", flush=True)
            except Exception as e:
                print(f"t={t}: {type(e).__name__}: {e}, skipped", flush=True)
    else:
        ctx = mp.get_context("fork")
        if threadpool_limits is None:
            print("note: threadpoolctl not found -> workers may oversubscribe threads.", flush=True)
        print(f"Extracting {len(TIMES)} frames across {N_WORKERS} workers ...", flush=True)
        with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx,
                                 initializer=_init_worker, initargs=(THREADS_PER_WORKER,)) as ex:
            futs = {ex.submit(extract, t): t for t in TIMES}
            for fut in as_completed(futs):
                t = futs[fut]
                try:
                    results.append(fut.result()); print(f"t={t}: done", flush=True)
                except FileNotFoundError:
                    print(f"t={t}: missing file, skipped", flush=True)
                except Exception as e:
                    print(f"t={t}: {type(e).__name__}: {e}, skipped", flush=True)

    if not results:
        raise SystemExit("No frames extracted (check paths / TIMES).")
    frames = sorted(results, key=lambda r: r[0])

    # ---- fixed scales ----
    # left: colour scale for |J_parallel|
    vmax = max(float(np.percentile(J2, 99)) for _, J2, _, _, _, _ in frames)
    # right: log-spaced bins + robust y-limit
    allv = np.concatenate([np.concatenate([f[3], f[4], f[5]]) for f in frames])
    allv = allv[allv > 0]
    if allv.size:
        bins = np.logspace(np.log10(allv.min()), np.log10(allv.max()), NBINS)
    else:
        bins = np.logspace(-1, 1, NBINS)          # no sheets anywhere: harmless placeholder axis
    hist_peaks = []
    for _, _, _, m1, m2, m3 in frames:
        for arr in (m1, m2, m3):
            if len(arr) >= MIN_N_SCALE:
                hist_peaks.append(np.histogram(arr, bins=bins, density=True)[0].max())
    ymax_hist = (np.percentile(hist_peaks, YMAX_PCTILE) if hist_peaks else 1.0) * 1.1

    series = [("M1", "C0"), ("M2", "C1"), ("M3", "C2")]

    # physical axes (pixels -> d_i): extent 0 .. N*dx  (= 0 .. 256 for a 4096 grid, dx=0.0625)
    Nx, Ny = frames[0][1].shape
    extent = [0, Nx * dx, 0, Ny * dy]
    xc = (np.arange(Nx) + 0.5) * dx        # pixel-centre coords, aligned with imshow
    yc = (np.arange(Ny) + 0.5) * dy
    Xc, Yc = np.meshgrid(xc, yc)           # shape (Ny, Nx) -> matches mask.T for the contour

    # ---- two-panel figure ----
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

    # LEFT: image updated via set_data (axes never cleared -> colourbar stays linked)
    im0 = axL.imshow(frames[0][1].T, cmap="inferno", origin="lower", vmin=0, vmax=vmax,
                     extent=extent)
    fig.colorbar(im0, ax=axL, label=r"$|J_\parallel|$")
    axL.set_xlim(0, Nx * dx); axL.set_ylim(0, Ny * dy)
    axL.set_xlabel(r"$x/d_p$"); axL.set_ylabel(r"$y/d_p$")
    axL.set_title("current-sheet detection")
    cs_holder = {"cs": None}

    def update(i):
        t, J2, mask, m1, m2, m3 = frames[i]

        # LEFT: |J_parallel| map + sheet outline
        im0.set_data(J2.T)
        if cs_holder["cs"] is not None:
            try:
                cs_holder["cs"].remove()
            except Exception:
                for coll in cs_holder["cs"].collections:
                    coll.remove()
        # only contour if there is a sheet to outline (empty mask -> no contour)
        if mask.any():
            cs_holder["cs"] = axL.contour(Xc, Yc, mask.T, levels=[0.5],
                                          colors="cyan", linewidths=0.6)
        else:
            cs_holder["cs"] = None

        # RIGHT: thickness histogram
        axR.clear()
        any_data = False
        for arr, (name, col) in zip((m1, m2, m3), series):
            if len(arr):
                any_data = True
                axR.hist(arr, bins=bins, density=True, histtype="step", lw=2.5, color=col,
                         label=fr"{name}: {np.median(arr):.2f} $d_i$")
                axR.axvline(np.median(arr), color=col, ls=":", lw=1.2)
        axR.set_xscale("log"); axR.set_xlim(bins[0], bins[-1]); axR.set_ylim(0, ymax_hist)
        axR.set_xlabel(r"thickness $(d_i)$"); axR.set_ylabel("PDF")
        axR.set_title("thickness distribution")
        axR.axvline(x=0.858, linestyle='--', color='green', label='median of CS thickness')
        axR.axvline(x=9.16, linestyle='--', color='cyan', label='median of CS length')
        
        if any_data:                                  # no sheets -> skip legend (avoids warning)
            axR.legend(loc="upper right")

        fig.suptitle(rf"$\beta_p = {beta}$,    $\beta_e = 1$,    $t = {t}$")
        return []

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps)

    try:
        anim.save(out_movie, writer="ffmpeg", fps=fps, dpi=120)
        print(f"saved {out_movie}", flush=True)
    except Exception as e:
        out_movie = out_movie.replace(".mp4", ".gif")
        anim.save(out_movie, writer="pillow", fps=fps, dpi=100)
        print(f"ffmpeg unavailable ({type(e).__name__}); saved {out_movie}", flush=True)
    plt.close(fig)

Extracting 301 frames across 128 workers ...
t=0: done
t=1: done
t=3: done
t=2: done
t=6: done
t=7: done
t=4: done
t=8: done
t=5: done
t=9: done
t=10: done
t=13: done
t=11: done
t=12: done
t=14: done
t=16: done
t=15: done
t=17: done
t=18: done
t=19: done
t=20: done
t=22: done
t=21: done
t=23: done
t=25: done
t=24: done
t=27: done
t=26: done
t=28: done
t=29: done
t=31: done
t=30: done
t=32: done
t=33: done
t=34: done
t=35: done
t=36: done
t=37: done
t=38: done
t=39: done
t=40: done
t=42: done
t=41: done
t=43: done
t=45: done
t=44: done
t=46: done
t=47: done
t=48: done
t=49: done
t=51: done
t=52: done
t=53: done
t=50: done
t=55: done
t=54: done
t=59: done
t=57: done
t=56: done
t=60: done
t=61: done
t=58: done
t=62: done
t=66: done
t=63: done
t=64: done
t=65: done
t=67: done
t=70: done
t=71: done
t=72: done
t=68: done
t=75: done
t=69: done
t=73: done
t=76: done
t=77: done
t=81: done
t=74: done
t=80: done
t=83: done
t=78: done
t=79: done
t=82: done
t=84: done
t=88: done
t=87: done
t=98: do